In [2]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-05 13:55:30.238902: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-05 13:55:35.894484: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [13]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 1,
        "ips": ['172.190.116.144'],
        "ports": [50151]
      }
      
    },
  "temp_data_path": "../../../",
  "partitions": 1,
  "iterations": 1,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [5]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [6]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [14]:
X_train, y_train = get_train_data()

print(type(X_train))

<class 'numpy.ndarray'>


In [8]:
model = create_model()
rain = Rain(config, model)

2023-07-05 13:56:04,339 [ERROR] [Rain] Error in the config: Error in partitions: argument of type 'int' is not iterable
2023-07-05 13:56:04,342 [DEBUG] [Rain] Rain is initialized
2023-07-05 13:56:04,344 [DEBUG] [Provisioner] Creating coordinator
2023-07-05 13:56:04,345 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-05 13:56:04,347 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-05 13:56:04,348 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-05 13:56:04,349 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 13:56:04,352 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 13:56:04,354 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [9]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-05 13:56:07,521 [INFO] [Provisioner] provisioner is serving
2023-07-05 13:56:07,523 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 13:56:07,526 [INFO] [Coordinator] coordinator is serving
2023-07-05 13:56:07,527 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 13:56:07,545 [DEBUG] [Provisioner] Received 'NumOfWorkers: 1
' from the coordinator to define the number of workers
2023-07-05 13:56:07,547 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 13:56:07,562 [DEBUG] [LocalProvisioner] Creating 1 workers
2023-07-05 13:56:07,563 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 13:56:07,566 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 13:56:07,568 [DEBUG] [Provisioner] [Created workers]
IPs : ['127.0.0.1'], ports: [50151], statuses: [1], IDs : [1]
2023-07-05 13:56:07,572 [DEBUG] [DividerAmbassador] divider ambassador is serving


469/469 [==============================] - 6s 9ms/step - loss: 0.4303 - accuracy: 0.8673


2023-07-05 13:57:06,281 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 13:57:06,283 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider


2023-07-05 13:57:06,586 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-05 13:57:06,607 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
2023-07-05 13:57:06,673 [DEBUG] [DeepLearning] Iteration 1/1 complete for worker 1.
2023-07-05 13:57:06,695 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-05 13:57:06,697 [DEBUG] [Divider] Divider stopped serving
2023-07-05 13:57:06,698 [INFO] [Worker_50151] Worker stopped serving on port: 50151
2023-07-05 13:57:06,699 [DEBUG] [LocalProvisioner] Workers are deleted
2023-07-05 13:57:06,702 [INFO] [Provisioner] provisioner stopped serving


In [15]:
print(type(model))

<class 'keras.engine.sequential.Sequential'>


In [10]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

2023-07-05 13:57:37.969688: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 31360000 exceeds 10% of free system memory.


79/79 [==============================] - 1s 4ms/step - loss: 0.1485 - accuracy: 0.9519

Test accuracy: 95.2%


In [11]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-05 13:57:40,613 [INFO] [Provisioner] provisioner is serving
2023-07-05 13:57:40,614 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 13:57:40,616 [INFO] [Coordinator] coordinator is serving
2023-07-05 13:57:40,617 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 13:57:40,621 [DEBUG] [Provisioner] Received 'NumOfWorkers: 1
' from the coordinator to define the number of workers
2023-07-05 13:57:40,623 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 13:57:40,624 [DEBUG] [LocalProvisioner] Creating 1 workers
2023-07-05 13:57:40,626 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 13:57:40,627 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 13:57:40,627 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 13:57:40,629 [DEBUG] [Provisioner] [Created workers]
IPs : ['127.0.0.1'], ports: [50151], statuses: [1], IDs : [1]
202

469/469 [==============================] - 5s 9ms/step - loss: 0.1924 - accuracy: 0.9426


2023-07-05 13:58:39,182 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 13:58:39,186 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider


2023-07-05 13:58:39,494 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-05 13:58:39,556 [DEBUG] [DeepLearning] Iteration 1/1 complete.
DEBUG:DeepLearning:Iteration 1/1 complete.
2023-07-05 13:58:39,564 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-05 13:58:39,568 [DEBUG] [Divider] Divider stopped serving
DEBUG:Divider:Divider stopped serving
2023-07-05 13:58:39,571 [INFO] [Worker_50151] Worker stopped serving on port: 50151
2023-07-05 13:58:39,571 [INFO] [Worker_50151] Worker stopped serving on port: 50151
INFO:Worker_50151:Worker stopped serving on port: 50151
2023-07-05 13:58:39,575 [INFO] [Worker_50151] Worker stopped serving on port: 50151
2023-07-05 13:58:39,575 [INFO] [Worker_50151] Worker stopped serving on port: 50151
INFO:Wor

In [12]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

2023-07-05 13:58:39.621175: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 31360000 exceeds 10% of free system memory.


79/79 [==============================] - 1s 3ms/step - loss: 0.1028 - accuracy: 0.9669

Test accuracy: 96.7%
